# Sign and verify a deployment: the attestation manifest

When a controller runs on a robot near people, one question matters: **exactly
what is running, and can you prove it?** jaxility answers with an **attestation
manifest** — a content-addressed record that hash-chains the robot's calibrated
identity to the deployed binary. It is the supply-chain receipt for a robot
controller.

The chain:

```
 robot telemetry hash ─► Jaxterity attestation handle ─► jaxility manifest ─► deployed binary hash
   (what was measured)      (the calibrated robot)         (toolchain+target)    (the actual .so)
```

Verify the manifest and you confirm: this binary was built from *this* robot, with
*this* toolchain, for *this* target — and nothing was altered. Change any link and
the verification fails. This tutorial builds a real manifest, **tampers with it**,
and watches the check catch it.

In [1]:
import os, platform, ctypes
from pathlib import Path
def _preload_acados():
    root=os.environ.get("ACADOS_SOURCE_DIR")
    if not root or platform.system()!="Darwin": return
    for n in ("libblasfeo.dylib","libhpipm.dylib","libqpOASES_e.dylib","libacados.dylib"):
        p=Path(root)/"lib"/n
        if p.exists():
            try: ctypes.CDLL(str(p))
            except OSError: pass
_preload_acados()
import numpy as np, jax.numpy as jnp, tempfile
print("acados", "set" if os.environ.get("ACADOS_SOURCE_DIR") else "MISSING (build cells fail)")

acados set


## Step 1 — Build a real manifest

We build the Cartpole controller (as in tutorial #1) and inspect the manifest it
carries. Every field that enters the content hash is a link in the chain.

In [2]:
import jaxterity.zoo as zoo
from jaxterity.zoo.cartpole import reduced_params
from jaxility.lowering import translate
from jaxility.templates import lqr
from jaxility.builder import build_for_target
from jaxility.targets import current_host_target
from jaxility.manifest import verify_manifest

NX,NU = 4,1; INITIAL_STATE=(0.3,0.0,0.0,0.0)
def build(robot, name):
    p=reduced_params(robot)
    def dyn(s,u):
        g,mp,mc,L=p["g"],p["mp"],p["mc"],p["L"]; th,xd,thd=s[1],s[2],s[3]
        si,co=jnp.sin(th),jnp.cos(th); den=mc+mp*si*si
        return jnp.array([xd,thd,(u[0]+mp*si*(L*thd**2+g*co))/den,
            (-u[0]*co-mp*L*thd**2*co*si-(mc+mp)*g*si)/(L*den)])
    cf=translate(dyn, in_shapes=((NX,),(NU,)), name=name)
    spec=lqr(cf,Q=(10.,10.,1.,1.),R=(0.1,),initial_state=INITIAL_STATE,
             input_bounds=((-20.,),(20.,)),name=name,horizon_steps=20,time_horizon_s=1.0)
    return build_for_target(dynamics=cf, spec=spec, target=current_host_target(),
        source_attestation_handle=bytes.fromhex(robot.attestation_handle),
        work_dir=Path(tempfile.mkdtemp()))

robot = zoo.load("cartpole").with_provenance(("attest-tut","v0","telemetry-hash"), calibrated=True)
bundle = build(robot, "attest_cartpole")
m = bundle.manifest
print("manifest links (the hash-chain inputs):")
print("  source_attestation_handle :", m.source_attestation_handle.hex()[:24], "…  (the robot)")
print("  target_profile_hash       :", m.target_profile_hash.hex()[:24], "…  (the chip)")
print("  toolchain_versions        :", dict(list(m.toolchain_versions.items())[:2]), "…")
print("  artifact_content_hash     :", m.artifact_content_hash.hex()[:24], "…  (the .so)")
print("  content_hash (the chain)  :", m.content_hash().hex()[:24], "…")

manifest links (the hash-chain inputs):
  source_attestation_handle : 6c750219b1b09b3726137c3d …  (the robot)
  target_profile_hash       : d0504997b908ba4179d4b040 …  (the chip)
  toolchain_versions        : {'clang': 'apple-clang-21', 'acados-template': '0.5.1'} …
  artifact_content_hash     : 53347829a6918359f4d7b0d0 …  (the .so)
  content_hash (the chain)  : 90d16c7b21216d66d8bda259 …


## Step 2 — Verify it (and tamper with it)

`verify_manifest` recomputes the content hash and reports a structured verdict.
With the *real* expected hash it passes; with a *wrong* one (a stand-in for "the
registry says this deployment should be X, but the artifact is Y") it fails and
says why. There is no silent pass.

In [3]:
# (a) honest verification against the registry's expected hash
true_hash = m.content_hash()
rep_ok = verify_manifest(m, expected_content_hash=true_hash)
print("(a) honest:   ok =", rep_ok.ok, "|", rep_ok.reason)
print("    signature_status:", rep_ok.signature_status)

# (b) the registry expected a DIFFERENT deployment -> caught
wrong = bytearray(true_hash); wrong[0] ^= 0x01   # flip one bit of the expected hash
rep_bad = verify_manifest(m, expected_content_hash=bytes(wrong))
print("\n(b) mismatch: ok =", rep_bad.ok, "|", rep_bad.reason[:90])

(a) honest:   ok = True | schema v0; content hash recomputed; signer accepted.
    signature_status: absent

(b) mismatch: ok = False | recomputed content hash differs from the expected hash — the manifest has been tampered wi


In [4]:
# (c) tamper with the artifact itself: a different binary -> a different chain.
import blake3
from jaxility.manifest import Manifest
forged = m.model_copy(update={"artifact_content_hash": blake3.blake3(b"a-swapped-binary").digest()})
print("(c) forged-artifact manifest content_hash == original?",
      forged.content_hash() == m.content_hash())
print("    -> swapping the binary changes the chain; a registry pinned to the")
print("       original hash rejects it.")

(c) forged-artifact manifest content_hash == original? False
    -> swapping the binary changes the chain; a registry pinned to the
       original hash rejects it.


## Step 3 — Recalibrate the robot → the whole chain moves

Because the chain is *rooted on the robot*, recalibrating it (here: a 2× pole
mass) re-roots everything: the attestation handle, the artifact, and the manifest
content hash all change. The receipt tracks the physics.

In [5]:
heavy = robot.with_parameters({"pole.mass": robot.parameters()["pole.mass"]*2.0})
hb = build(heavy, "attest_cartpole_heavy")
print("attestation handle moved :", robot.attestation_handle != heavy.attestation_handle)
print("artifact hash moved      :", bundle.artifact.content_hash != hb.artifact.content_hash)
print("manifest content moved   :", bundle.manifest.content_hash() != hb.manifest.content_hash())

attestation handle moved : True
artifact hash moved      : True
manifest content moved   : True


## Step 4 — Dual-path: cover *both* artifacts

A dual-path deployment (tutorial #3) ships two artifacts — the controller and the
learned policy — under a safety envelope. `attest_dual_path` binds all three into
one verifiable hash, so retraining the policy or loosening the envelope moves the
attestation just as recalibration does.

In [6]:
from jaxility.compose import attest_dual_path, CompositionPlan, SafetyEnvelope
env = SafetyEnvelope(name="cp", state_lower=(-5.,-3.,-20.,-20.), state_upper=(5.,3.,20.,20.),
                     input_lower=(-20.,), input_upper=(20.,))
plan = CompositionPlan(name="cp-dual", mpc_period_ns=1_000_000, policy_period_ns=20_000_000,
                       safety_envelope=env)
a1 = attest_dual_path(controller_manifest=m, policy_artifact_bytes=b"policy-weights-v1", plan=plan)
a2 = attest_dual_path(controller_manifest=m, policy_artifact_bytes=b"policy-weights-v2", plan=plan)  # retrained
print("dual-path attestation hash:", a1.content_hash().hex()[:24], "…")
print("  it binds: controller manifest + policy artifact + composition plan")
print("retrain the policy -> attestation moves:", a1.content_hash() != a2.content_hash())

dual-path attestation hash: 2def5353c5ba0041ec3c7eaf …
  it binds: controller manifest + policy artifact + composition plan
retrain the policy -> attestation moves: True


## Step 5 — Signing: OSS hash-chain today, real signatures tomorrow

At the OSS level the manifest is an *unsigned* BLAKE3 hash chain — reproducibility
and tamper-evidence without key management (`signature_status: absent`). The
`Signer` interface is pluggable: an enterprise build swaps in real signing
(Sigstore/SLSA-aligned) without changing the schema. So the audit story scales
from "anyone can recompute and check the hash" to "cryptographically signed by an
identity" with the same manifest.

In [7]:
print("OSS manifest — signer_identity:", m.signer_identity, "| signature:", m.signature)
print("verifier verdict signature_status:", rep_ok.signature_status,
      "(unsigned hash-chain; an enterprise Signer would report 'verified')")

OSS manifest — signer_identity: None | signature: None
verifier verdict signature_status: absent (unsigned hash-chain; an enterprise Signer would report 'verified')


## Recap

The attestation manifest is the **provenance receipt** for a deployed controller:

- a **BLAKE3 hash chain** from the robot's calibrated identity to the deployed
  binary — toolchain and target included;
- **tamper-evident** — change the binary, the robot, the policy, or the safety
  envelope, and verification fails;
- **deterministic** — same source + toolchain + target → the same hash, regardless
  of when it was built;
- and **signer-pluggable** — unsigned hash-chain at the OSS level, real signatures
  at the enterprise tier, one schema.

This is what lets every other tutorial say "we deployed *exactly* this" and mean
it. Next: **calibrate a real robot from telemetry** — produce the very robot
identity this chain is rooted on, from your LeKiwi's measured motion.